# Sequence Prediction - STRICT protocol

Strict copy of `_results_sequence_prediction.ipynb`. Every stage runs with
`--strict`, which switches the pipeline from our permissive default to the
evaluation population Screen2Vec itself uses in `modeltester.py`:

| | permissive (default notebook) | `--strict` (this notebook) |
|---|---|---|
| traces kept | >= 3 screens | >= `n_predictors + 1` (= 5) |
| context per prediction | 1..4 screens (truncated near trace start) | exactly 4 |
| candidate pool | every embedding in the directory | only screens of eligible traces |
| top-k% thresholds | `ceil()` | `int()` truncation, as in `modeltester.py:141` |
| training negatives | any screen | only screens of eligible traces |

Because every training window is full under `--strict`, the zero-padding
branch in the loss never fires, so training and evaluation see the same
context length. That mismatch is present in the permissive pipeline.

**Artefacts are suffixed `_strict`** and cannot clobber the permissive run:

    embeddings_seq_768_strict/                    (9_)
    lstm_predictor_768d_strict.pth                (9_)
    sequence_projector_768d_strict.pth            (9_)
    embeddings_seq_768_strict_with_app_desc/      (6_)

`6_generate_app_context_embeddings.py` needs no `--strict` of its own - it just
concatenates an app vector per screen, so it is pointed at the strict
directories instead.

### Two caveats on the numbers this produces

1. **`--strict` cannot reproduce Screen2Vec's screen *ordering*.** The original
   builds each trace from an unsorted `os.listdir()` of `view_hierarchies`
   (`UI_embedding/precompute_embeddings.py:72`), which is not temporal order and
   is not reproducible on another filesystem. This pipeline always uses
   `gestures.json` order, i.e. true temporal order.

2. **Screen-ID namespace bug, unresolved.** Embeddings are keyed by Rico's
   *global* `data/layouts/` numbering, while `gestures.json` keys are *per-trace*
   indices. They are different namespaces, so every trace is currently matched to
   unrelated screens (verified: 0 of 731 sampled trace screens match
   `layouts/<id>.json`). `--strict` does not fix this and inherits it. Treat the
   output below as protocol-comparable, not yet correct.

In [ ]:
# Pipeline v2 - strict

In [ ]:
%run 9_train_joint.py --proj_dim 768 --strict

In [ ]:
%run 6_generate_app_context_embeddings.py --embeddings_dir embeddings_seq_768_strict --output_dir embeddings_seq_768_strict_with_app_desc

In [ ]:
%run 4_evaluate_screen_prediction.py --embeddings_dir embeddings_seq_768_strict_with_app_desc --lstm_model lstm_predictor_768d_strict.pth --strict